# Modular Backtesting Framework
## Master Control Panel & Strategy Optimizer

In [1]:
# =============================================================================
# CELL 1 - CONFIGURATION (Master Control Panel)
# =============================================================================

# --- Library Imports ---
import pandas as pd
import vectorbt as vbt
import numpy as np
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings('ignore')

# =============================================================================
# FEATURE FLAGS (Enable/Disable Indicators)
# Set to True to include in optimization, False to exclude
# =============================================================================

USE_VOLUME_FILTER = True
USE_RSI = True
USE_STOCH_RSI = False
USE_AROON = False
USE_BOLLINGER = False     # ← Disable (conflicts with trend following)
USE_CCI = False
USE_SCHAFF_TREND = False
USE_KAMA = False
USE_ALMA = False
USE_MACD = False
USE_ADX = False
USE_OBV = False
USE_CMF = False
USE_ATR = False
USE_BB_WIDTH = False      # ← Disable (too restrictive)

# =============================================================================
# BASE STRATEGY PARAMETERS (Always Active - Triple EMA Crossover)
# =============================================================================

fast_range = range(4, 20, 4)     # Fast EMA periods: 4, 9, 14, 19, 24, 29, 34, 39
med_range = range(50, 90, 10)     # Medium EMA periods: 50, 55, 60, 65, 70, 75, 80, 85
slow_range = range(120, 250, 10)  # Slow EMA periods: 120, 125, ... 245



# =============================================================================
# INDICATOR PARAMETER RANGES
# =============================================================================

# --- TREND INDICATORS ---
# Schaff Trend Cycle: Combines MACD with Stochastic for smoother trend detection
schaff_cycle_range = range(20, 50, 10)  # Cycle period: 20, 30, 40

# KAMA: Adapts to market volatility - faster in trends, slower in consolidation
kama_range = range(10, 30, 5)  # Period: 10, 15, 20, 25

# ALMA: Gaussian-weighted MA that reduces lag while maintaining smoothness
alma_range = range(9, 50, 10)        # Window: 9, 19, 29, 39, 49
alma_offset_range = [0.85, 0.90, 0.95]  # Offset controls weight distribution
alma_sigma_range = [4, 6, 8]            # Sigma controls smoothness

# MACD: Classic momentum indicator using EMA crossovers
macd_fast_range = range(8, 16, 2)    # Fast EMA: 8, 10, 12, 14
macd_slow_range = range(20, 30, 2)   # Slow EMA: 20, 22, 24, 26, 28
macd_signal_range = range(7, 11, 2)  # Signal line: 7, 9

# ADX: Measures trend strength regardless of direction
adx_period_range = range(10, 20, 5)  # Period: 10, 15
adx_threshold_range = [20, 25, 30]   # Minimum ADX for trend confirmation

# --- MEAN REVERSION INDICATORS ---
# RSI: Classic overbought/oversold oscillator
rsi_window_range = range(7, 22, 7)   # Window: 7, 9, 11, 13, 15, 17, 19
rsi_oversold_range = [20, 30, 40]    # Oversold threshold (buy signal)
rsi_overbought_range = [50, 55, 60]  # Overbought threshold (sell signal)
rsi_threshold_range = [40, 45, 50, 55]


# Stochastic RSI: RSI applied to RSI for more sensitivity
stoch_rsi_window_range = range(10, 20, 5)  # Window: 10, 15
stoch_rsi_threshold_range = [20, 30, 40]   # Oversold threshold

# Aroon: Identifies trend changes and strength using highs/lows
aroon_period_range = range(20, 30, 5)  # Period: 20, 25
aroon_threshold_range = [50, 70, 90]   # Up-Down difference threshold

# Bollinger Bands: Statistical bands around moving average
bb_window_range = range(15, 25, 5)  # Window: 15, 20
bb_std_range = [1.5, 2.0, 2.5]      # Standard deviation multiplier

# CCI: Measures price deviation from statistical mean
cci_window_range = range(14, 28, 7)  # Window: 14, 21
cci_threshold_range = [100, 150, 200]  # Extreme threshold

# --- VOLUME INDICATORS ---
# Volume Filter: Basic volume spike detection
vol_ma_range = range(10, 70, 5)       # Volume MA period: 10, 15, ... 65
vol_mult_range = np.arange(0.7, 1.4, 0.1)  # Volume multiplier threshold

# OBV: Cumulative volume based on price direction
obv_ma_range = range(10, 30, 5)  # OBV smoothing period: 10, 15, 20, 25

# CMF: Measures buying/selling pressure using volume and price location
cmf_period_range = range(10, 30, 10)  # Period: 10, 20
cmf_threshold_range = [0.0, 0.05, 0.1]  # Positive flow threshold

# --- VOLATILITY INDICATORS ---
# ATR: Measures market volatility using true range
atr_period_range = range(10, 20, 5)  # Period: 10, 15
atr_mult_range = [1.0, 1.5, 2.0]     # ATR multiplier for filtering

# BB Width: Bollinger Band expansion/contraction
bb_width_threshold_range = [0.01, 0.02, 0.03]  # Minimum band width

# =============================================================================
# DATA & SPLIT SETTINGS
# =============================================================================

# DATA_PATH = r"C:\Users\mkova\OneDrive\Počítač\autism\BTCUSD_1d_Binance.csv"
TRAIN_SPLIT_RATIO = 0.6  # 60% train, 40% validation

# =============================================================================
# BACKTEST SETTINGS
# =============================================================================

batch_size = 100       # Number of strategies per batch (memory management)
init_cash = 100_000    # Initial portfolio value
fees = 0.0005          # Trading fees (0.05%)

# =============================================================================
# HARDCODED INDICATOR DEFAULTS (used when indicator is proxy/placeholder)
# =============================================================================

SCHAFF_MACD_FAST = 12     # Schaff Trend Cycle uses MACD as proxy
SCHAFF_MACD_SLOW = 26
SCHAFF_MACD_SIGNAL = 9

BB_WIDTH_WINDOW = 20      # BB Width default Bollinger params
BB_WIDTH_ALPHA = 2.0

ATR_ROLLING_WINDOW = 50   # ATR mean calculation window

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

# Collect active filters
active_filters = []
if USE_VOLUME_FILTER: active_filters.append('VOLUME_FILTER')
if USE_RSI: active_filters.append('RSI')
if USE_STOCH_RSI: active_filters.append('STOCH_RSI')
if USE_AROON: active_filters.append('AROON')
if USE_SCHAFF_TREND: active_filters.append('SCHAFF_TREND')
if USE_KAMA: active_filters.append('KAMA')
if USE_ALMA: active_filters.append('ALMA')
if USE_MACD: active_filters.append('MACD')
if USE_ADX: active_filters.append('ADX')
if USE_BOLLINGER: active_filters.append('BOLLINGER')
if USE_CCI: active_filters.append('CCI')
if USE_OBV: active_filters.append('OBV')    
if USE_CMF: active_filters.append('CMF')
if USE_ATR: active_filters.append('ATR')
if USE_BB_WIDTH: active_filters.append('BB_WIDTH')

# Count total parameter ranges
param_count = 3  # Base EMA ranges
if USE_VOLUME_FILTER: param_count += 2
if USE_RSI: param_count += 3
if USE_STOCH_RSI: param_count += 2
if USE_AROON: param_count += 2
if USE_SCHAFF_TREND: param_count += 1
if USE_KAMA: param_count += 1
if USE_ALMA: param_count += 3
if USE_MACD: param_count += 3
if USE_ADX: param_count += 2
if USE_BOLLINGER: param_count += 2
if USE_CCI: param_count += 2
if USE_OBV: param_count += 1
if USE_CMF: param_count += 2
if USE_ATR: param_count += 2
if USE_BB_WIDTH: param_count += 1

print("✅ Configuration loaded")
print(f"Active filters: {active_filters if active_filters else ['None (Pure EMA Strategy)']}")
print(f"Total parameter ranges defined: {param_count}")

# Warning about combination explosion
if len(active_filters) > 3:
    print()
    print(f"⚠️  WARNING: {len(active_filters)} indicators enabled - combinations may explode!")
    print("   Consider reducing parameter ranges or disabling some indicators.")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def extract_best_params(params_tuple):
    """Extract best parameters from tuple into a dictionary."""
    p = {}
    p['fast_ema'] = params_tuple[0]
    p['med_ema'] = params_tuple[1]
    p['slow_ema'] = params_tuple[2]
    idx = 3
    
    if USE_VOLUME_FILTER:
        p['vol_window'] = params_tuple[idx]
        p['vol_mult'] = params_tuple[idx + 1]
        idx += 2
    if USE_RSI:
        p['rsi_window'] = params_tuple[idx]
        p['rsi_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_STOCH_RSI:
        p['stoch_window'] = params_tuple[idx]
        p['stoch_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_AROON:
        p['aroon_period'] = params_tuple[idx]
        p['aroon_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_SCHAFF_TREND:
        p['schaff_cycle'] = params_tuple[idx]
        idx += 1
    if USE_KAMA:
        p['kama_period'] = params_tuple[idx]
        idx += 1
    if USE_ALMA:
        p['alma_window'] = params_tuple[idx]
        p['alma_offset'] = params_tuple[idx + 1]
        p['alma_sigma'] = params_tuple[idx + 2]
        idx += 3
    if USE_MACD:
        p['macd_fast'] = params_tuple[idx]
        p['macd_slow'] = params_tuple[idx + 1]
        p['macd_signal'] = params_tuple[idx + 2]
        idx += 3
    if USE_ADX:
        p['adx_period'] = params_tuple[idx]
        p['adx_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_BOLLINGER:
        p['bb_window'] = params_tuple[idx]
        p['bb_std'] = params_tuple[idx + 1]
        idx += 2
    if USE_CCI:
        p['cci_window'] = params_tuple[idx]
        p['cci_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_OBV:
        p['obv_ma'] = params_tuple[idx]
        idx += 1
    if USE_CMF:
        p['cmf_period'] = params_tuple[idx]
        p['cmf_threshold'] = params_tuple[idx + 1]
        idx += 2
    if USE_ATR:
        p['atr_period'] = params_tuple[idx]
        p['atr_mult'] = params_tuple[idx + 1]
        idx += 2
    if USE_BB_WIDTH:
        p['bb_width_threshold'] = params_tuple[idx]
        idx += 1
    return p

def calculate_signals(close_data, volume_data, high_data, low_data, params):
    """Calculate entry and exit signals for given data and parameters."""
    # Calculate base EMAs
    ema_f = vbt.MA.run(close_data, window=params['fast_ema'], ewm=True).ma.squeeze()
    ema_m = vbt.MA.run(close_data, window=params['med_ema'], ewm=True).ma.squeeze()
    ema_s = vbt.MA.run(close_data, window=params['slow_ema'], ewm=True).ma.squeeze()
    
    # Base trend condition
    trend = (ema_f > ema_m) | (ema_m > ema_s)
    entries = trend.copy()
    
    # Store indicators for visualization
    indicators = {'ema_f': ema_f, 'ema_m': ema_m, 'ema_s': ema_s}
    
    # Apply all active indicator conditions
    if USE_VOLUME_FILTER:
        vol_ma = vbt.MA.run(volume_data, window=params['vol_window']).ma.squeeze()
        vol_cond = volume_data > (vol_ma * params['vol_mult'])
        entries = entries & vol_cond
        indicators['vol_ma'] = vol_ma
    
    if USE_RSI:
        rsi = vbt.RSI.run(close_data, window=params['rsi_window']).rsi.squeeze()
        rsi_cond = rsi > params['rsi_threshold']
        entries = entries & rsi_cond
        indicators['rsi'] = rsi
    
    if USE_STOCH_RSI:
        stoch = vbt.STOCHRSI.run(close_data, window=params['stoch_window']).stochrsi.squeeze() * 100
        stoch_cond = stoch < params['stoch_threshold']
        entries = entries & stoch_cond
        indicators['stoch'] = stoch
    
    if USE_AROON:
        aroon = vbt.AROON.run(high_data, low_data, window=params['aroon_period'])
        aroon_diff = aroon.aroon_up.squeeze() - aroon.aroon_down.squeeze()
        aroon_cond = aroon_diff > params['aroon_threshold']
        entries = entries & aroon_cond
        indicators['aroon_diff'] = aroon_diff
    
    if USE_SCHAFF_TREND:
        schaff = vbt.MACD.run(close_data, fast_window=SCHAFF_MACD_FAST, slow_window=SCHAFF_MACD_SLOW, signal_window=SCHAFF_MACD_SIGNAL)
        schaff_cond = schaff.macd.squeeze() > 0
        entries = entries & schaff_cond
        indicators['schaff'] = schaff
    
    if USE_KAMA:
        kama = vbt.KAMA.run(close_data, window=params['kama_period']).kama.squeeze()
        kama_cond = close_data > kama
        entries = entries & kama_cond
        indicators['kama'] = kama
    
    if USE_ALMA:
        alma = vbt.MA.run(close_data, window=params['alma_window']).ma.squeeze()
        alma_cond = close_data > alma
        entries = entries & alma_cond
        indicators['alma'] = alma
    
    if USE_MACD:
        macd = vbt.MACD.run(close_data, fast_window=params['macd_fast'], slow_window=params['macd_slow'], signal_window=params['macd_signal'])
        macd_cond = macd.macd.squeeze() > macd.signal.squeeze()
        entries = entries & macd_cond
        indicators['macd'] = macd
    
    if USE_ADX:
        adx = vbt.ADX.run(high_data, low_data, close_data, window=params['adx_period']).adx.squeeze()
        adx_cond = adx > params['adx_threshold']
        entries = entries & adx_cond
        indicators['adx'] = adx
    
    if USE_BOLLINGER:
        bb = vbt.BBANDS.run(close_data, window=params['bb_window'], alpha=params['bb_std'])
        bb_cond = close_data < bb.lower.squeeze()
        entries = entries & bb_cond
        indicators['bb'] = bb
    
    if USE_CCI:
        cci = vbt.CCI.run(high_data, low_data, close_data, window=params['cci_window']).cci.squeeze()
        cci_cond = cci < -params['cci_threshold']
        entries = entries & cci_cond
        indicators['cci'] = cci
    
    if USE_OBV:
        obv = vbt.OBV.run(close_data, volume_data).obv.squeeze()
        obv_ma = vbt.MA.run(obv, window=params['obv_ma']).ma.squeeze()
        obv_cond = obv > obv_ma
        entries = entries & obv_cond
        indicators['obv'] = obv
        indicators['obv_ma'] = obv_ma
    
    if USE_CMF:
        cmf = vbt.CMF.run(high_data, low_data, close_data, volume_data, window=params['cmf_period']).cmf.squeeze()
        cmf_cond = cmf > params['cmf_threshold']
        entries = entries & cmf_cond
        indicators['cmf'] = cmf
    
    if USE_ATR:
        atr = vbt.ATR.run(high_data, low_data, close_data, window=params['atr_period']).atr.squeeze()
        atr_mean = atr.rolling(ATR_ROLLING_WINDOW).mean()
        atr_cond = atr > (atr_mean * params['atr_mult'])
        entries = entries & atr_cond
        indicators['atr'] = atr
        indicators['atr_mean'] = atr_mean
    
    if USE_BB_WIDTH:
        bb_w = vbt.BBANDS.run(close_data, window=BB_WIDTH_WINDOW, alpha=BB_WIDTH_ALPHA)
        bb_width = (bb_w.upper.squeeze() - bb_w.lower.squeeze()) / bb_w.middle.squeeze()
        bbw_cond = bb_width > params['bb_width_threshold']
        entries = entries & bbw_cond
        indicators['bb_width'] = bb_width
    
    # Exit signal: Fast EMA crosses below Medium EMA
    exits = ema_f < ema_m
    
    # Clean signals: only take first entry after each exit
    entries_clean = entries.vbt.signals.first(after=exits)
    
    return entries_clean, exits, indicators

def run_portfolio(close_data, entries, exits, force_close_last=False):
    """Run portfolio simulation with given signals."""
    exits_final = exits.copy()
    if force_close_last:
        exits_final.iloc[-1] = True
    return vbt.Portfolio.from_signals(
        close_data,
        entries,
        exits_final,
        init_cash=init_cash,
        fees=fees,
        freq='D'
    )

print("\n✅ Helper functions loaded")

✅ Configuration loaded
Active filters: ['VOLUME_FILTER', 'RSI']
Total parameter ranges defined: 8

✅ Helper functions loaded


In [2]:
# =============================================================================
# CELL 2 - DATA LOADING (Multi-Asset via yfinance API)
# =============================================================================

# Define assets to fetch
TICKERS = ['TQQQ', 'GLD', 'BTC-USD']
START_DATE = '2015-01-01'
END_DATE = None  # None means fetch up to present

# Dictionary to store all asset data
asset_data = {}

print("📊 MULTI-ASSET DATA LOADED VIA YFINANCE")
print("=" * 50)

for ticker in TICKERS:
    try:
        # Fetch data from yfinance
        # Note: auto_adjust=True adjusts OHLC prices for dividends/splits but volume remains unadjusted
        data = yf.download(ticker, start=START_DATE, end=END_DATE, interval='1d', progress=False, auto_adjust=True)
        
        if data.empty:
            print(f"⚠️  Warning: No data found for {ticker}")
            continue
        
        # Handle MultiIndex columns (yfinance sometimes returns this)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        
        # Extract price and volume series
        close = data['Close'].astype(float)
        volume = data['Volume'].astype(float)
        high = data['High'].astype(float)
        low = data['Low'].astype(float)
        
        # Calculate split index
        split_idx = int(len(close) * TRAIN_SPLIT_RATIO)
        
        # Store data for this asset
        # Create safe key: replace special chars, lowercase, ensure valid Python identifier
        asset_key = ''.join(c if c.isalnum() else '_' for c in ticker).lower()
        asset_data[asset_key] = {
            'ticker': ticker,
            'close': close,
            'volume': volume,
            'high': high,
            'low': low,
            'split_idx': split_idx,
            'split_date': close.index[split_idx],
            'train_close': close.iloc[:split_idx],
            'train_volume': volume.iloc[:split_idx],
            'train_high': high.iloc[:split_idx],
            'train_low': low.iloc[:split_idx],
            'val_close': close.iloc[split_idx:],
            'val_volume': volume.iloc[split_idx:],
            'val_high': high.iloc[split_idx:],
            'val_low': low.iloc[split_idx:]
        }
        
        # Print summary for this asset
        total = len(close)
        train_count = len(asset_data[asset_key]['train_close'])
        val_count = len(asset_data[asset_key]['val_close'])
        
        print(f"Asset: {ticker}")
        print(f"  Range: {close.index[0].date()} to {close.index[-1].date()}")
        print(f"  Total: {total:,} samples")
        print(f"  Train: {train_count:,} ({train_count/total*100:.1f}%) | Validation: {val_count:,} ({val_count/total*100:.1f}%)")
        print()
        
    except Exception as e:
        print(f"❌ Error fetching {ticker}: {str(e)}")
        continue

# Create individual variables for each asset (for backwards compatibility)
# TQQQ data
if 'tqqq' in asset_data:
    tqqq_close = asset_data['tqqq']['close']
    tqqq_volume = asset_data['tqqq']['volume']
    tqqq_high = asset_data['tqqq']['high']
    tqqq_low = asset_data['tqqq']['low']
    train_tqqq_close = asset_data['tqqq']['train_close']
    train_tqqq_volume = asset_data['tqqq']['train_volume']
    train_tqqq_high = asset_data['tqqq']['train_high']
    train_tqqq_low = asset_data['tqqq']['train_low']
    val_tqqq_close = asset_data['tqqq']['val_close']
    val_tqqq_volume = asset_data['tqqq']['val_volume']
    val_tqqq_high = asset_data['tqqq']['val_high']
    val_tqqq_low = asset_data['tqqq']['val_low']

# GLD data
if 'gld' in asset_data:
    gld_close = asset_data['gld']['close']
    gld_volume = asset_data['gld']['volume']
    gld_high = asset_data['gld']['high']
    gld_low = asset_data['gld']['low']
    train_gld_close = asset_data['gld']['train_close']
    train_gld_volume = asset_data['gld']['train_volume']
    train_gld_high = asset_data['gld']['train_high']
    train_gld_low = asset_data['gld']['train_low']
    val_gld_close = asset_data['gld']['val_close']
    val_gld_volume = asset_data['gld']['val_volume']
    val_gld_high = asset_data['gld']['val_high']
    val_gld_low = asset_data['gld']['val_low']

# BTC-USD data
if 'btc_usd' in asset_data:
    btc_close = asset_data['btc_usd']['close']
    btc_volume = asset_data['btc_usd']['volume']
    btc_high = asset_data['btc_usd']['high']
    btc_low = asset_data['btc_usd']['low']
    train_btc_close = asset_data['btc_usd']['train_close']
    train_btc_volume = asset_data['btc_usd']['train_volume']
    train_btc_high = asset_data['btc_usd']['train_high']
    train_btc_low = asset_data['btc_usd']['train_low']
    val_btc_close = asset_data['btc_usd']['val_close']
    val_btc_volume = asset_data['btc_usd']['val_volume']
    val_btc_high = asset_data['btc_usd']['val_high']
    val_btc_low = asset_data['btc_usd']['val_low']

# Set default asset for existing backtest logic (use BTC-USD for backwards compatibility)
# This allows Cell 3 and Cell 4 to run without modification
if 'btc_usd' in asset_data:
    close = btc_close
    volume = btc_volume
    high = btc_high
    low = btc_low
    split_idx = asset_data['btc_usd']['split_idx']
    split_date = asset_data['btc_usd']['split_date']
    train_close = train_btc_close
    train_volume = train_btc_volume
    train_high = train_btc_high
    train_low = train_btc_low
    val_close = val_btc_close
    val_volume = val_btc_volume
    val_high = val_btc_high
    val_low = val_btc_low
    print("📌 Default asset for backtest: BTC-USD")
elif len(asset_data) > 0:
    # Fallback to first available asset
    default_key = list(asset_data.keys())[0]
    default_asset = asset_data[default_key]
    close = default_asset['close']
    volume = default_asset['volume']
    high = default_asset['high']
    low = default_asset['low']
    split_idx = default_asset['split_idx']
    split_date = default_asset['split_date']
    train_close = default_asset['train_close']
    train_volume = default_asset['train_volume']
    train_high = default_asset['train_high']
    train_low = default_asset['train_low']
    val_close = default_asset['val_close']
    val_volume = default_asset['val_volume']
    val_high = default_asset['val_high']
    val_low = default_asset['val_low']
    print(f"📌 Default asset for backtest: {default_asset['ticker']}")
else:
    print("❌ No asset data loaded! Check your internet connection and ticker symbols.")

In [3]:
# =============================================================================
# CELL 3 - BUILD COMBINATIONS
# =============================================================================

# Start with base EMA combinations (fast < medium < slow constraint)
combos = [
    (f, m, s) 
    for f in fast_range 
    for m in med_range 
    for s in slow_range 
    if f < m < s
]

print(f"Base EMA combinations: {len(combos):,}")

# Expand combinations based on enabled flags
# Order matters - must match unpacking order in CELL 4

if USE_VOLUME_FILTER:
    combos = [(*c, v_w, v_m) for c in combos for v_w in vol_ma_range for v_m in vol_mult_range]
    print(f"+ Volume Filter: {len(combos):,} combinations")

if USE_RSI:
    combos = [(*c, rsi_w, rsi_thresh) for c in combos 
              for rsi_w in rsi_window_range 
              for rsi_thresh in rsi_threshold_range]
    print(f"+ RSI: {len(combos):,} combinations")

if USE_STOCH_RSI:
    combos = [(*c, sr_w, sr_t) for c in combos 
              for sr_w in stoch_rsi_window_range 
              for sr_t in stoch_rsi_threshold_range]
    print(f"+ Stoch RSI: {len(combos):,} combinations")

if USE_AROON:
    combos = [(*c, ar_p, ar_t) for c in combos 
              for ar_p in aroon_period_range 
              for ar_t in aroon_threshold_range]
    print(f"+ Aroon: {len(combos):,} combinations")

if USE_SCHAFF_TREND:
    combos = [(*c, sc_c) for c in combos for sc_c in schaff_cycle_range]
    print(f"+ Schaff Trend: {len(combos):,} combinations")

if USE_KAMA:
    combos = [(*c, k_p) for c in combos for k_p in kama_range]
    print(f"+ KAMA: {len(combos):,} combinations")

if USE_ALMA:
    combos = [(*c, a_w, a_o, a_s) for c in combos 
              for a_w in alma_range 
              for a_o in alma_offset_range 
              for a_s in alma_sigma_range]
    print(f"+ ALMA: {len(combos):,} combinations")

if USE_MACD:
    combos = [(*c, m_f, m_s, m_sig) for c in combos 
              for m_f in macd_fast_range 
              for m_s in macd_slow_range 
              for m_sig in macd_signal_range]
    print(f"+ MACD: {len(combos):,} combinations")

if USE_ADX:
    combos = [(*c, adx_p, adx_t) for c in combos 
              for adx_p in adx_period_range 
              for adx_t in adx_threshold_range]
    print(f"+ ADX: {len(combos):,} combinations")

if USE_BOLLINGER:
    combos = [(*c, bb_w, bb_s) for c in combos 
              for bb_w in bb_window_range 
              for bb_s in bb_std_range]
    print(f"+ Bollinger: {len(combos):,} combinations")

if USE_CCI:
    combos = [(*c, cci_w, cci_t) for c in combos 
              for cci_w in cci_window_range 
              for cci_t in cci_threshold_range]
    print(f"+ CCI: {len(combos):,} combinations")

if USE_OBV:
    combos = [(*c, obv_m) for c in combos for obv_m in obv_ma_range]
    print(f"+ OBV: {len(combos):,} combinations")

if USE_CMF:
    combos = [(*c, cmf_p, cmf_t) for c in combos 
              for cmf_p in cmf_period_range 
              for cmf_t in cmf_threshold_range]
    print(f"+ CMF: {len(combos):,} combinations")

if USE_ATR:
    combos = [(*c, atr_p, atr_m) for c in combos 
              for atr_p in atr_period_range 
              for atr_m in atr_mult_range]
    print(f"+ ATR: {len(combos):,} combinations")

if USE_BB_WIDTH:
    combos = [(*c, bbw_t) for c in combos for bbw_t in bb_width_threshold_range]
    print(f"+ BB Width: {len(combos):,} combinations")

# Unpack combinations into separate parameter lists using zip
# Build the unpacking dynamically based on enabled flags
param_names = ['fast_periods', 'med_periods', 'slow_periods']

if USE_VOLUME_FILTER:
    param_names.extend(['vol_windows', 'vol_mults'])
if USE_RSI:
    param_names.extend(['rsi_windows', 'rsi_oversolds', 'rsi_overboughts'])
if USE_STOCH_RSI:
    param_names.extend(['stoch_windows', 'stoch_thresholds'])
if USE_AROON:
    param_names.extend(['aroon_periods', 'aroon_thresholds'])
if USE_SCHAFF_TREND:
    param_names.extend(['schaff_cycles'])
if USE_KAMA:
    param_names.extend(['kama_periods'])
if USE_ALMA:
    param_names.extend(['alma_windows', 'alma_offsets', 'alma_sigmas'])
if USE_MACD:
    param_names.extend(['macd_fasts', 'macd_slows', 'macd_signals'])
if USE_ADX:
    param_names.extend(['adx_periods', 'adx_thresholds'])
if USE_BOLLINGER:
    param_names.extend(['bb_windows', 'bb_stds'])
if USE_CCI:
    param_names.extend(['cci_windows', 'cci_thresholds'])
if USE_OBV:
    param_names.extend(['obv_mas'])
if USE_CMF:
    param_names.extend(['cmf_periods', 'cmf_thresholds'])
if USE_ATR:
    param_names.extend(['atr_periods', 'atr_mults'])
if USE_BB_WIDTH:
    param_names.extend(['bb_width_thresholds'])

# Unpack all parameters
unpacked = list(zip(*combos))

# Assign to named lists
fast_periods = list(unpacked[0])
med_periods = list(unpacked[1])
slow_periods = list(unpacked[2])

idx = 3
if USE_VOLUME_FILTER:
    vol_windows = list(unpacked[idx])
    vol_mults = list(unpacked[idx + 1])
    idx += 2
if USE_RSI:
    rsi_windows = list(unpacked[idx])
    rsi_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_STOCH_RSI:
    stoch_windows = list(unpacked[idx])
    stoch_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_AROON:
    aroon_periods = list(unpacked[idx])
    aroon_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_SCHAFF_TREND:
    schaff_cycles = list(unpacked[idx])
    idx += 1
if USE_KAMA:
    kama_periods = list(unpacked[idx])
    idx += 1
if USE_ALMA:
    alma_windows = list(unpacked[idx])
    alma_offsets = list(unpacked[idx + 1])
    alma_sigmas = list(unpacked[idx + 2])
    idx += 3
if USE_MACD:
    macd_fasts = list(unpacked[idx])
    macd_slows = list(unpacked[idx + 1])
    macd_signals = list(unpacked[idx + 2])
    idx += 3
if USE_ADX:
    adx_periods = list(unpacked[idx])
    adx_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_BOLLINGER:
    bb_windows = list(unpacked[idx])
    bb_stds = list(unpacked[idx + 1])
    idx += 2
if USE_CCI:
    cci_windows = list(unpacked[idx])
    cci_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_OBV:
    obv_mas = list(unpacked[idx])
    idx += 1
if USE_CMF:
    cmf_periods = list(unpacked[idx])
    cmf_thresholds = list(unpacked[idx + 1])
    idx += 2
if USE_ATR:
    atr_periods = list(unpacked[idx])
    atr_mults = list(unpacked[idx + 1])
    idx += 2
if USE_BB_WIDTH:
    bb_width_thresholds = list(unpacked[idx])
    idx += 1

# Count active filters
count_active = len(active_filters)

print()
print("=" * 50)
print(f"🔬 Testing {len(combos):,} strategies with {count_active} active filters")

# Memory warning for large combinations
if len(combos) > 100000:
    print(f"⚠️  Large search space! Consider increasing batch_size or reducing parameters.")

Base EMA combinations: 208
+ Volume Filter: 17,472 combinations
+ RSI: 209,664 combinations

🔬 Testing 209,664 strategies with 2 active filters
⚠️  Large search space! Consider increasing batch_size or reducing parameters.


In [4]:
# =============================================================================
# CELL 4 - BATCHED BACKTEST
# =============================================================================

# Initialize result storage
all_sharpe = []
all_returns = []
all_combos = []

# Process in batches for memory efficiency
for i in tqdm(range(0, len(combos), batch_size), desc="Batch Progress"):
    
    # Extract batch slice
    batch_combos = combos[i:i + batch_size]
    batch_fast = fast_periods[i:i + batch_size]
    batch_med = med_periods[i:i + batch_size]
    batch_slow = slow_periods[i:i + batch_size]
    
    # Extract batch parameters for enabled indicators
    if USE_VOLUME_FILTER:
        batch_vol_windows = vol_windows[i:i + batch_size]
        batch_vol_mults = vol_mults[i:i + batch_size]
    if USE_RSI:
        batch_rsi_windows = rsi_windows[i:i + batch_size]
        batch_rsi_threshold = rsi_thresholds[i:i + batch_size]
    if USE_STOCH_RSI:
        batch_stoch_windows = stoch_windows[i:i + batch_size]
        batch_stoch_threshold = stoch_thresholds[i:i + batch_size]
    if USE_AROON:
        batch_aroon_periods = aroon_periods[i:i + batch_size]
        batch_aroon_threshold = aroon_thresholds[i:i + batch_size]
    if USE_SCHAFF_TREND:
        batch_schaff_cycles = schaff_cycles[i:i + batch_size]
    if USE_KAMA:
        batch_kama_periods = kama_periods[i:i + batch_size]
    if USE_ALMA:
        batch_alma_windows = alma_windows[i:i + batch_size]
        batch_alma_offsets = alma_offsets[i:i + batch_size]
        batch_alma_sigmas = alma_sigmas[i:i + batch_size]
    if USE_MACD:
        batch_macd_fasts = macd_fasts[i:i + batch_size]
        batch_macd_slows = macd_slows[i:i + batch_size]
        batch_macd_signals = macd_signals[i:i + batch_size]
    if USE_ADX:
        batch_adx_periods = adx_periods[i:i + batch_size]
        batch_adx_threshold = adx_thresholds[i:i + batch_size]
    if USE_BOLLINGER:
        batch_bb_windows = bb_windows[i:i + batch_size]
        batch_bb_stds = bb_stds[i:i + batch_size]
    if USE_CCI:
        batch_cci_windows = cci_windows[i:i + batch_size]
        batch_cci_threshold = cci_thresholds[i:i + batch_size]
    if USE_OBV:
        batch_obv_mas = obv_mas[i:i + batch_size]
    if USE_CMF:
        batch_cmf_periods = cmf_periods[i:i + batch_size]
        batch_cmf_threshold = cmf_thresholds[i:i + batch_size]
    if USE_ATR:
        batch_atr_periods = atr_periods[i:i + batch_size]
        batch_atr_mults = atr_mults[i:i + batch_size]
    if USE_BB_WIDTH:
        batch_bb_width_threshold = bb_width_thresholds[i:i + batch_size]
    
    # Validate window sizes
    if any(w <= 0 for w in batch_fast + batch_med + batch_slow):
        continue
    
    # =================================================================
    # BASE STRATEGY: Triple EMA Crossover
    # Entry: Fast > Medium > Slow (uptrend alignment)
    # Exit: Fast < Medium (trend weakening)
    # =================================================================
    
    # Calculate EMAs for the batch
    ema_f = vbt.MA.run(train_close, window=batch_fast, ewm=True).ma
    ema_m = vbt.MA.run(train_close, window=batch_med, ewm=True).ma
    ema_s = vbt.MA.run(train_close, window=batch_slow, ewm=True).ma
    
    # Set column names to combo tuples for tracking
    ema_f.columns = batch_combos
    ema_m.columns = batch_combos
    ema_s.columns = batch_combos
    
    # Base trend condition: EMAs in bullish alignment
    trend = (ema_f > ema_m) & (ema_m > ema_s)
    
    # Initialize entries with base trend
    entries = trend.copy()
    
    # =================================================================
    # CONDITIONAL FILTERS - Applied based on feature flags
    # =================================================================
    
    # --- VOLUME FILTER ---
    # Requires volume to be above its moving average * multiplier
    if USE_VOLUME_FILTER:
        vol_ma = vbt.MA.run(train_volume, window=batch_vol_windows).ma
        vol_ma.columns = batch_combos
        vol_raw = pd.DataFrame(
            np.tile(train_volume.values.reshape(-1, 1), len(batch_combos)),
            index=train_volume.index,
            columns=batch_combos
        )
        mult_arr = pd.DataFrame(
            np.tile(batch_vol_mults, (len(train_volume), 1)),
            index=train_volume.index,
            columns=batch_combos
        )
        vol_condition = vol_raw > (vol_ma * mult_arr)
        entries = entries & vol_condition
    
    # --- RSI FILTER ---
    # Requires RSI to be in oversold territory for entry
    if USE_RSI:
        rsi = vbt.RSI.run(train_close, window=batch_rsi_windows).rsi
        rsi.columns = batch_combos
        rsi_thresh_arr = pd.DataFrame(
            np.tile(batch_rsi_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        rsi_condition = rsi > rsi_thresh_arr
        entries = entries & rsi_condition   
    
    # --- STOCHASTIC RSI FILTER ---
    # More sensitive than regular RSI, catches momentum shifts earlier
    if USE_STOCH_RSI:
        stoch_rsi = vbt.STOCHRSI.run(train_close, window=batch_stoch_windows).stochrsi
        stoch_rsi.columns = batch_combos
        stoch_thresh_arr = pd.DataFrame(
            np.tile(batch_stoch_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        stoch_condition = (stoch_rsi * 100) < stoch_thresh_arr
        entries = entries & stoch_condition
    
    # --- AROON FILTER ---
    # Measures time since highest high vs lowest low
    if USE_AROON:
        aroon = vbt.AROON.run(train_high, train_low, window=batch_aroon_periods)
        aroon_up = aroon.aroon_up
        aroon_down = aroon.aroon_down
        aroon_up.columns = batch_combos
        aroon_down.columns = batch_combos
        aroon_thresh_arr = pd.DataFrame(
            np.tile(batch_aroon_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        aroon_condition = (aroon_up - aroon_down) > aroon_thresh_arr
        entries = entries & aroon_condition
    
    # --- SCHAFF TREND CYCLE ---
    # Combines MACD with double-smoothed stochastic
    # Placeholder: Implement custom Schaff calculation or use external library
    if USE_SCHAFF_TREND:
        # Note: VectorBT doesn't have built-in Schaff, using MACD as proxy
        # For full implementation, calculate: MACD -> Stochastic -> Smooth -> Stochastic -> Smooth
        schaff_proxy = vbt.MACD.run(train_close, fast_window=12, slow_window=26, signal_window=9)
        schaff_base = (schaff_proxy.macd.squeeze() > 0).values
        schaff_condition = pd.DataFrame(
            np.tile(schaff_base.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        entries = entries & schaff_condition
    
    # --- KAMA FILTER ---
    # Adaptive MA that adjusts to volatility
    if USE_KAMA:
        kama = vbt.KAMA.run(train_close, window=batch_kama_periods).kama
        kama.columns = batch_combos
        price_broadcast = pd.DataFrame(
            np.tile(train_close.values.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        kama_condition = price_broadcast > kama
        entries = entries & kama_condition
    
    # --- ALMA FILTER ---
    # Gaussian-weighted MA with offset and sigma parameters
    # Placeholder: VectorBT doesn't have built-in ALMA
    if USE_ALMA:
        # Using SMA as placeholder - for full implementation use custom ALMA calculation
        alma_proxy = vbt.MA.run(train_close, window=batch_alma_windows).ma
        alma_proxy.columns = batch_combos
        price_broadcast = pd.DataFrame(
            np.tile(train_close.values.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        alma_condition = price_broadcast > alma_proxy
        entries = entries & alma_condition
    
    # --- MACD FILTER ---
    # Classic momentum: MACD line above signal line
    if USE_MACD:
        macd = vbt.MACD.run(
            train_close, 
            fast_window=batch_macd_fasts, 
            slow_window=batch_macd_slows, 
            signal_window=batch_macd_signals
        )
        macd_line = macd.macd
        signal_line = macd.signal
        macd_line.columns = batch_combos
        signal_line.columns = batch_combos
        macd_condition = macd_line > signal_line
        entries = entries & macd_condition
    
    # --- ADX FILTER ---
    # Requires strong trend (ADX above threshold)
    if USE_ADX:
        adx = vbt.ADX.run(train_high, train_low, train_close, window=batch_adx_periods).adx
        adx.columns = batch_combos
        adx_thresh_arr = pd.DataFrame(
            np.tile(batch_adx_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        adx_condition = adx > adx_thresh_arr
        entries = entries & adx_condition
    
    # --- BOLLINGER BANDS FILTER ---
    # Mean reversion: price below lower band
    if USE_BOLLINGER:
        bb = vbt.BBANDS.run(train_close, window=batch_bb_windows, alpha=batch_bb_stds)
        bb_lower = bb.lower
        bb_lower.columns = batch_combos
        price_broadcast = pd.DataFrame(
            np.tile(train_close.values.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        bb_condition = price_broadcast < bb_lower
        entries = entries & bb_condition
    
    # --- CCI FILTER ---
    # Oversold condition: CCI below negative threshold
    if USE_CCI:
        cci = vbt.CCI.run(train_high, train_low, train_close, window=batch_cci_windows).cci
        cci.columns = batch_combos
        cci_thresh_arr = pd.DataFrame(
            np.tile(batch_cci_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        cci_condition = cci < -cci_thresh_arr
        entries = entries & cci_condition
    
    # --- OBV FILTER ---
    # Volume confirmation: OBV above its moving average
    if USE_OBV:
        obv = vbt.OBV.run(train_close, train_volume).obv
        obv_ma_calc = vbt.MA.run(obv, window=batch_obv_mas).ma
        obv_broadcast = pd.DataFrame(
            np.tile(obv.values.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        obv_ma_calc.columns = batch_combos
        obv_condition = obv_broadcast > obv_ma_calc
        entries = entries & obv_condition
    
    # --- CMF FILTER ---
    # Money flow: CMF above threshold indicates buying pressure
    if USE_CMF:
        cmf = vbt.CMF.run(train_high, train_low, train_close, train_volume, window=batch_cmf_periods).cmf
        cmf.columns = batch_combos
        cmf_thresh_arr = pd.DataFrame(
            np.tile(batch_cmf_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        cmf_condition = cmf > cmf_thresh_arr
        entries = entries & cmf_condition
    
    # --- ATR FILTER ---
    # Volatility expansion: ATR above rolling mean
    if USE_ATR:
        atr = vbt.ATR.run(train_high, train_low, train_close, window=batch_atr_periods).atr
        atr.columns = batch_combos
        atr_mean = atr.rolling(50).mean()
        atr_mult_arr = pd.DataFrame(
            np.tile(batch_atr_mults, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        atr_condition = atr > (atr_mean * atr_mult_arr)
        entries = entries & atr_condition
    
    # --- BB WIDTH FILTER ---
    # Volatility: Band width above threshold (avoiding squeeze)
    if USE_BB_WIDTH:
        bb_for_width = vbt.BBANDS.run(train_close, window=20, alpha=2.0)
        bb_width_vals = ((bb_for_width.upper.squeeze() - bb_for_width.lower.squeeze()) / bb_for_width.middle.squeeze()).values
        bb_width_broadcast = pd.DataFrame(
            np.tile(bb_width_vals.reshape(-1, 1), len(batch_combos)),
            index=train_close.index,
            columns=batch_combos
        )
        bbw_thresh_arr = pd.DataFrame(
            np.tile(batch_bb_width_threshold, (len(train_close), 1)),
            index=train_close.index,
            columns=batch_combos
        )
        bb_width_condition = bb_width_broadcast > bbw_thresh_arr
        entries = entries & bb_width_condition
    
    # =================================================================
    # EXIT SIGNAL: Fast EMA crosses below Medium EMA
    # =================================================================
    exits = (ema_f < ema_m) 
    
    # Clean signals: only take first entry after each exit
    entries = entries.vbt.signals.first(after=exits)
    
    # =================================================================
    # RUN PORTFOLIO SIMULATION
    # =================================================================
    pf = vbt.Portfolio.from_signals(
        train_close,
        entries,
        exits,
        init_cash=init_cash,
        fees=fees,
        freq='D'
    )
    
    # Store results
    all_sharpe.extend(pf.sharpe_ratio().values)
    all_returns.extend(pf.total_return().values)
    all_combos.extend(batch_combos)

print()
print("=" * 50)
print(f"✅ Grid Search Complete. Tested {len(all_combos):,} strategies.")

Batch Progress:   0%|          | 0/2097 [00:00<?, ?it/s]

Batch Progress: 100%|██████████| 2097/2097 [02:29<00:00, 13.99it/s]


✅ Grid Search Complete. Tested 209,664 strategies.


In [5]:
print(all_combos[70000])

(8, 60, 160, 35, 0.8999999999999999, 14, 40)


In [6]:
# =============================================================================
# CELL 5 - RESULTS & ANALYSIS
# =============================================================================

# Calculate benchmark (buy and hold)
bench_train = vbt.Portfolio.from_holding(
    train_close, 
    init_cash=init_cash, 
    fees=fees, 
    freq='D'
)
benchmark_return = bench_train.total_return()

# Check if any strategies were tested
if len(all_combos) == 0:
    print("❌ No strategies were tested! Check your data and parameters.")
    raise SystemExit

# Find best strategy by Sharpe ratio
all_sharpe_clean = [s if not np.isnan(s) else -np.inf for s in all_sharpe]
best_idx = np.argmax(all_sharpe_clean)
best_params = all_combos[best_idx]
best_sharpe = all_sharpe[best_idx]
best_return = all_returns[best_idx]

# =================================================================
# PRINT RESULTS
# =================================================================

print("🏆 BEST STRATEGY PARAMETERS")
print("=" * 50)
print(f"EMA: Fast={best_params[0]}, Medium={best_params[1]}, Slow={best_params[2]}")

# Print parameters for each enabled indicator
param_idx = 3

if USE_VOLUME_FILTER:
    print(f"Volume Filter: Window={best_params[param_idx]}, Multiplier={best_params[param_idx+1]:.2f}")
    param_idx += 2

if USE_RSI:
    print(f"RSI: Window={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_STOCH_RSI:
    print(f"Stochastic RSI: Window={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_AROON:
    print(f"Aroon: Period={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_SCHAFF_TREND:
    print(f"Schaff Trend Cycle: Cycle={best_params[param_idx]}")
    param_idx += 1

if USE_KAMA:
    print(f"KAMA: Period={best_params[param_idx]}")
    param_idx += 1

if USE_ALMA:
    print(f"ALMA: Window={best_params[param_idx]}, Offset={best_params[param_idx+1]}, Sigma={best_params[param_idx+2]}")
    param_idx += 3

if USE_MACD:
    print(f"MACD: Fast={best_params[param_idx]}, Slow={best_params[param_idx+1]}, Signal={best_params[param_idx+2]}")
    param_idx += 3

if USE_ADX:
    print(f"ADX: Period={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_BOLLINGER:
    print(f"Bollinger Bands: Window={best_params[param_idx]}, Std={best_params[param_idx+1]}")
    param_idx += 2

if USE_CCI:
    print(f"CCI: Window={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_OBV:
    print(f"OBV: MA Period={best_params[param_idx]}")
    param_idx += 1

if USE_CMF:
    print(f"CMF: Period={best_params[param_idx]}, Threshold={best_params[param_idx+1]}")
    param_idx += 2

if USE_ATR:
    print(f"ATR: Period={best_params[param_idx]}, Multiplier={best_params[param_idx+1]}")
    param_idx += 2

if USE_BB_WIDTH:
    print(f"BB Width: Threshold={best_params[param_idx]}")
    param_idx += 1

print()
print("📊 PERFORMANCE METRICS")
print("=" * 50)
print(f"Sharpe Ratio: {best_sharpe:.4f}")
print(f"Total Return: {best_return*100:.2f}%")
print(f"Benchmark Return (Buy & Hold): {benchmark_return*100:.2f}%")
print(f"Outperformance: {(best_return - benchmark_return)*100:.2f}%")

# Additional statistics
print()
print("📈 OPTIMIZATION STATISTICS")
print("=" * 50)
valid_sharpes = [s for s in all_sharpe if not np.isnan(s) and not np.isinf(s)]
print(f"Strategies with valid Sharpe: {len(valid_sharpes):,} / {len(all_sharpe):,}")
if len(valid_sharpes) > 0:
    print(f"Mean Sharpe: {np.mean(valid_sharpes):.4f}")
    print(f"Max Sharpe: {np.max(valid_sharpes):.4f}")
    print(f"Min Sharpe: {np.min(valid_sharpes):.4f}")
else:
    print("No valid strategies found - try relaxing filter conditions.")

🏆 BEST STRATEGY PARAMETERS
EMA: Fast=16, Medium=50, Slow=140
Volume Filter: Window=10, Multiplier=0.70
RSI: Window=21, Threshold=55

📊 PERFORMANCE METRICS
Sharpe Ratio: 1.3884
Total Return: 997.09%
Benchmark Return (Buy & Hold): 404.41%
Outperformance: 592.68%

📈 OPTIMIZATION STATISTICS
Strategies with valid Sharpe: 209,664 / 209,664
Mean Sharpe: 0.9605
Max Sharpe: 1.3884
Min Sharpe: 0.5676
